# 🇧🇯 Bénin Insights Challenge — Notebook Data Analyste
## iSHEERO × DataCamp Donates 2026

---

**Rôle :** Data Analyste  
**Source :** GDELT — Global Database of Events, Language and Tone  
**Période :** Janvier 2025 – Décembre 2025  
**Données :** 10 722 événements réels GDELT sur le Bénin  
**Usage IA :** Structuration du notebook avec Claude (Anthropic). Analyses et insights : équipe.

---

### 📋 Mes 5 questions analytiques officielles (Q6 à Q10)

| # | Question | Colonnes utilisées |
|---|----------|-------------------|
| Q6 | Évolution du Score de Goldstein — stabilité ou tension ? | GoldsteinScale, MONTHYEAR |
| Q7 | Quels pays parlent du Bénin — regard positif ou négatif ? | Actor1CountryCode, AvgTone |
| Q8 | Pics d'événements autour de dates clés béninoises 2025 | SQLDATE, NumMentions, NumArticles |
| Q9 | Types d'événements dominants par région (nord vs sud) | EventRootCode, ActionGeo_FullName |
| Q10 | Ton médiatique selon l'origine du média | SOURCEURL, AvgTone, Actor1CountryCode |

In [ ]:
!pip install pandas plotly --quiet
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')
print('✅ Prêt !')

In [ ]:
# Chargement des vraies données GDELT
df = pd.read_csv('donnees_benin.csv', low_memory=False)
df['date'] = pd.to_datetime(df['SQLDATE'].astype(str), format='%Y%m%d', errors='coerce')
df = df.dropna(subset=['date'])
df['mois'] = df['date'].dt.to_period('M').astype(str)

# Codes CAMEO → labels lisibles
CAMEO = {1:'Déclarations',2:'Appels',3:'Intentions coopération',4:'Consultations',
         5:'Diplomatie',6:'Coopération matérielle',7:'Aide humanitaire',
         8:'Yield/Concessions',9:'Enquêtes',10:'Revendications',
         11:'Rejets/Refus',12:'Accusations',13:'Protestations',
         14:'Manifestations',15:'Menaces',16:'Sanctions',
         17:'Arrestations',18:'Violences verbales',19:'Violences armées'}

# Codes pays GDELT → noms
PAYS = {'BEN':'🇧🇯 Bénin','NGA':'🇳🇬 Nigeria','FRA':'🇫🇷 France',
        'AFR':'🌍 Afrique (général)','WAF':'🌍 Afrique de l\'Ouest',
        'NER':'🇳🇪 Niger','BFA':'🇧🇫 Burkina Faso','TGO':'🇹🇬 Togo',
        'GBR':'🇬🇧 Royaume-Uni','USA':'🇺🇸 États-Unis','CHN':'🇨🇳 Chine',
        'SEN':'🇸🇳 Sénégal','CIV':"🇨🇮 Côte d'Ivoire",'GHA':'🇬🇭 Ghana'}

print(f'✅ Données chargées !')
print(f'📊 Événements    : {len(df):,}')
print(f'📅 Période       : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'⚖️  Goldstein moy : {df["GoldsteinScale"].mean():.2f}/10')
print(f'🎭 Ton moyen     : {df["AvgTone"].mean():.2f}')
df.head(3)

---
## ⚖️ Q6 — Évolution du Score de Goldstein
**Le Bénin est-il en stabilité ou en tension sur 2025 ?**

In [ ]:
# Q6 : Score de Goldstein mensuel
df_gold = df.groupby('mois').agg(
    score_moy=('GoldsteinScale','mean'),
    nb_evt=('GoldsteinScale','count')
).reset_index().sort_values('mois')
df_gold['tendance'] = df_gold['score_moy'].rolling(3,center=True,min_periods=1).mean()
score_global = df['GoldsteinScale'].mean()

fig6 = go.Figure()
fig6.add_trace(go.Bar(
    x=df_gold['mois'], y=df_gold['score_moy'], name='Score mensuel',
    marker_color=['rgba(232,85,85,0.5)' if v<0 else 'rgba(29,158,117,0.5)' for v in df_gold['score_moy']],
    hovertemplate='<b>%{x}</b><br>Score : %{y:.2f}<extra></extra>'
))
fig6.add_trace(go.Scatter(
    x=df_gold['mois'], y=df_gold['tendance'],
    mode='lines+markers', line=dict(color='#534AB7',width=3),
    marker=dict(size=8), name='Tendance (moy. mobile 3 mois)'
))
fig6.add_hline(y=0,line_dash='dash',line_color='gray',
    annotation_text='Neutralité (0)',annotation_position='top right')
fig6.add_hline(y=score_global,line_dash='dot',line_color='orange',
    annotation_text=f'Moyenne 2025 : {score_global:.2f}',annotation_position='bottom right')

# Annoter dates clés 2025
dates_cles_2025 = {
    '2025-04':'🗳️ Préparatifs élections',
    '2025-08':'🎂 Fête indépendance',
    '2025-10':'⚠️ Tensions sécuritaires',
}
for mois_cle, label in dates_cles_2025.items():
    row = df_gold[df_gold['mois']==mois_cle]
    if len(row)>0:
        fig6.add_annotation(
            x=mois_cle, y=row['score_moy'].values[0],
            text=label, showarrow=True, arrowhead=2,
            bgcolor='#FEF3F2', bordercolor='#E85555',
            font=dict(size=10,color='#E85555'), ay=-50
        )

fig6.update_layout(
    title='⚖️ Q6 — Score de Goldstein mensuel au Bénin (Janv–Déc 2025)<br><sub>Données réelles GDELT · Rouge=instabilité · Vert=coopération</sub>',
    xaxis_title='Mois 2025', yaxis_title='Score Goldstein (-10 à +10)',
    template='plotly_white', height=480, xaxis_tickangle=-45,
    legend=dict(orientation='h',yanchor='bottom',y=1.02)
)
fig6.show()

pire = df_gold.loc[df_gold['score_moy'].idxmin()]
meilleur = df_gold.loc[df_gold['score_moy'].idxmax()]
print(f'📊 Score global 2025 : {score_global:.2f}/10 → {"Contexte STABLE ✅" if score_global>0 else "Contexte INSTABLE ⚠️"}')
print(f'📉 Mois le plus instable : {pire["mois"]} (score : {pire["score_moy"]:.2f})')
print(f'📈 Mois le plus stable   : {meilleur["mois"]} (score : {meilleur["score_moy"]:.2f})')

### 💬 Commentaire analytique — Q6
> **Complète avec tes observations :**  
> Le score de Goldstein moyen sur 2025 est de **[valeur]/10**. Le mois de **[mois]** enregistre le score le plus bas ([valeur]), probablement lié à **[événement béninois]**. La tendance montre que le Bénin est globalement en zone **[stable/instable]**.

---
## 🌍 Q7 — Quels pays parlent du Bénin ?
**Le regard international est-il positif ou négatif ?**

In [ ]:
# Q7 : Pays × Ton médiatique
df_pays = df.groupby('Actor1CountryCode').agg(
    nb_evt=('AvgTone','count'),
    ton_moy=('AvgTone','mean'),
    gold_moy=('GoldsteinScale','mean')
).reset_index()
df_pays = df_pays[df_pays['nb_evt']>=10].nlargest(15,'nb_evt')
df_pays['Pays'] = df_pays['Actor1CountryCode'].map(PAYS).fillna('🌐 '+df_pays['Actor1CountryCode'])
df_pays = df_pays.sort_values('nb_evt',ascending=True)

# Barplot horizontal coloré par ton
fig7 = go.Figure(go.Bar(
    x=df_pays['nb_evt'], y=df_pays['Pays'],
    orientation='h',
    marker=dict(
        color=df_pays['ton_moy'],
        colorscale='RdYlGn',
        cmid=0,
        showscale=True,
        colorbar=dict(title='Ton moyen')
    ),
    text=[f"Ton: {t:.1f}" for t in df_pays['ton_moy']],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Événements: %{x}<br>Ton: %{text}<extra></extra>'
))
fig7.update_layout(
    title='🌍 Q7 — Top 15 pays qui parlent du Bénin (2025)<br><sub>Couleur = ton médiatique · Rouge=négatif · Vert=positif · Données GDELT</sub>',
    xaxis_title="Nombre d'événements",
    yaxis_title='Pays source',
    template='plotly_white', height=520,
    margin=dict(l=10,r=100,t=80,b=40)
)
fig7.show()

print('🏆 Top 5 pays qui parlent du Bénin :')
top5 = df_pays.nlargest(5,'nb_evt')
for _,r in top5.iterrows():
    s = '😊 Positif' if r['ton_moy']>0 else '😟 Négatif'
    print(f"   {r['Pays']}: {int(r['nb_evt'])} événements | Ton: {r['ton_moy']:.2f} → {s}")

### 💬 Commentaire analytique — Q7
> **Complète avec tes observations :**  
> **[Pays 1]** domine avec **[X]** événements. Son regard est **[positif/négatif]**. Le Nigeria, voisin immédiat, représente **[X%]** de la couverture régionale. Les puissances extérieures (France, USA) ont un ton **[plus/moins]** favorable que les pays africains.

---
## 📅 Q8 — Pics d'événements & dates clés 2025
**Y a-t-il des pics identifiables autour d'événements béninois ?**

In [ ]:
# Q8 : Timeline mensuelle + articles
df_monthly = df.groupby('mois').agg(
    nb_evt=('GLOBALEVENTID','count'),
    articles=('NumArticles','sum'),
    mentions=('NumMentions','sum'),
    tone=('AvgTone','mean')
).reset_index().sort_values('mois')

# Double axe : événements + articles
fig8 = go.Figure()
fig8.add_trace(go.Bar(
    x=df_monthly['mois'], y=df_monthly['nb_evt'],
    name="Nombre d'événements",
    marker_color='rgba(29,158,117,0.7)',
    hovertemplate='<b>%{x}</b><br>%{y} événements<extra></extra>'
))
fig8.add_trace(go.Scatter(
    x=df_monthly['mois'], y=df_monthly['articles'],
    mode='lines+markers', name='Articles publiés',
    line=dict(color='#F59E0B',width=2.5),
    marker=dict(size=8),
    yaxis='y2',
    hovertemplate='<b>%{x}</b><br>%{y} articles<extra></extra>'
))

# Dates clés 2025 béninoises
dates_cles = {
    '2025-01':'🗳️ Élections législatives',
    '2025-04':'📊 Résultats élections',
    '2025-08':'🎂 65e Fête indépendance',
    '2025-10':'⚠️ Pic sécuritaire nord',
    '2025-12':'📅 Bilan annuel',
}
for mois_c, label in dates_cles.items():
    row = df_monthly[df_monthly['mois']==mois_c]
    if len(row)>0:
        fig8.add_vline(x=mois_c,line_dash='dot',line_color='#E85555',line_width=1.5)
        fig8.add_annotation(
            x=mois_c, y=row['nb_evt'].values[0],
            text=label, showarrow=True, arrowhead=2,
            bgcolor='#FEF3F2', bordercolor='#E85555',
            font=dict(size=9,color='#E85555'), ay=-45
        )

fig8.update_layout(
    title='📅 Q8 — Volume médiatique mensuel avec dates clés béninoises 2025<br><sub>Barres=événements · Ligne=articles publiés · Données réelles GDELT</sub>',
    xaxis_title='Mois 2025',
    yaxis=dict(title="Nombre d'événements",titlefont=dict(color='#1D9E75')),
    yaxis2=dict(title='Articles publiés',titlefont=dict(color='#F59E0B'),
                overlaying='y',side='right'),
    template='plotly_white', height=480, xaxis_tickangle=-45,
    legend=dict(orientation='h',yanchor='bottom',y=1.02)
)
fig8.show()

print('📊 Top 3 mois les plus couverts :')
top3 = df_monthly.nlargest(3,'nb_evt')[['mois','nb_evt','articles','mentions']]
print(top3.to_string(index=False))

### 💬 Commentaire analytique — Q8
> **Complète avec tes observations :**  
> Le mois de **[mois]** concentre le plus grand nombre d'événements (**[X]**), correspondant aux **[élections législatives de janvier 2025]**. On observe un second pic en **[mois]** lié à **[événement]**.

---
## 🗺️ Q9 — Types d'événements par région
**Le nord sécuritaire vs le sud économique — les données confirment-elles ?**

In [ ]:
# Q9 : Types événements par zone géographique
NORD_LIEUX = ['Kandi','Natitingou','Parakou','Djougou','Malanville','Banikoara']
SUD_LIEUX  = ['Cotonou','Porto-Novo','Abomey','Ouidah','Lokossa','Abomey-Calavi']

def classifier_zone(nom):
    nom_str = str(nom)
    if any(v in nom_str for v in NORD_LIEUX): return '🔴 Nord (zone sécuritaire)'
    elif any(v in nom_str for v in SUD_LIEUX): return '🟢 Sud (zone économique)'
    else: return '⚪ Bénin (général)'

df['zone'] = df['ActionGeo_FullName'].apply(classifier_zone)
df['EventLabel'] = df['EventRootCode'].map(CAMEO).fillna('Code '+df['EventRootCode'].astype(str))

# Catégoriser en Coopération vs Conflit
df['categorie'] = df['EventRootCode'].apply(
    lambda x: '🔴 Conflit' if x>=13 else ('🟢 Coopération' if x<=7 else '🟡 Neutre')
)

df_zone = df.groupby(['zone','categorie']).size().reset_index(name='count')
totals = df_zone.groupby('zone')['count'].transform('sum')
df_zone['pct'] = (df_zone['count']/totals*100).round(1)

fig9 = px.bar(
    df_zone, x='zone', y='pct', color='categorie',
    color_discrete_map={'🔴 Conflit':'#E85555','🟢 Coopération':'#1D9E75','🟡 Neutre':'#F59E0B'},
    title='🗺️ Q9 — Coopération vs Conflits par région au Bénin (2025)<br><sub>Données réelles GDELT · Nord=zone sécuritaire · Sud=zone économique</sub>',
    labels={'pct':'Proportion (%)','zone':'Région','categorie':'Catégorie'},
    text='pct', barmode='stack', height=480
)
fig9.update_traces(texttemplate='%{text:.1f}%',textposition='inside')
fig9.update_layout(template='plotly_white',legend_title='Catégorie')
fig9.show()

print('📊 Résumé par zone :')
for zone in df['zone'].unique():
    sub = df[df['zone']==zone]
    conflit_pct = (sub['EventRootCode']>=13).sum()/len(sub)*100
    print(f'   {zone}: {conflit_pct:.1f}% conflits | {100-conflit_pct:.1f}% coopération/neutre')

### 💬 Commentaire analytique — Q9
> **Complète avec tes observations :**  
> Le nord du Bénin enregistre **[X]%** de conflits contre **[Y]%** au sud. Cette différence de **[Z] points** confirme la pression sécuritaire dans la zone des trois frontières. Le sud reste dominé par des événements de coopération.

---
## 📰 Q10 — Ton médiatique selon l'origine du média
**Médias francophones vs anglophones vs africains — qui est le plus favorable au Bénin ?**

In [ ]:
# Q10 : Ton selon origine du média
def detecter_media(row):
    url = str(row['SOURCEURL']).lower()
    pays = str(row['Actor1CountryCode'])
    if any(d in url for d in ['rfi.fr','lemonde','fraternite.bj','24haubenin','matin-libre','beninwebtv']):
        return '🇫🇷 Médias francophones'
    elif any(d in url for d in ['bbc','reuters','apnews','voa','guardian']):
        return '🇬🇧 Médias anglophones'
    elif pays in ['BEN','TGO','NER','BFA','SEN','CIV','GHA','NGA','CMR']:
        return '🌍 Médias africains'
    elif pays in ['FRA','BEL','CHE','CAN']:
        return '🇫🇷 Médias francophones'
    elif pays in ['GBR','USA','AUS']:
        return '🇬🇧 Médias anglophones'
    elif pays in ['CHN','RUS']:
        return '🌏 Médias Chine/Russie'
    else:
        return '🌐 Autres'

df['espace_media'] = df.apply(detecter_media, axis=1)

df_media = df.groupby('espace_media').agg(
    nb=('AvgTone','count'),
    ton=('AvgTone','mean'),
    gold=('GoldsteinScale','mean')
).reset_index().sort_values('ton')

fig10 = px.box(
    df, x='espace_media', y='AvgTone',
    color='espace_media',
    title='📰 Q10 — Ton médiatique selon l\'espace linguistique (2025)<br><sub>Données réelles GDELT · Médiane = ligne centrale · 0 = neutralité</sub>',
    labels={'espace_media':'Espace médiatique','AvgTone':'Ton moyen (AvgTone)'},
    color_discrete_sequence=['#1D9E75','#3B7DD8','#F59E0B','#E85555','#534AB7'],
    height=500
)
fig10.add_hline(y=0,line_dash='dash',line_color='gray',
    annotation_text='Neutralité (0)',annotation_position='top right')
fig10.update_layout(template='plotly_white',showlegend=False,xaxis_tickangle=-15)
fig10.show()

print('📊 Ton moyen par espace médiatique :')
for _,r in df_media.iterrows():
    s = '😊 Favorable' if r['ton']>0 else '😟 Défavorable'
    print(f"   {r['espace_media']}: {r['ton']:.2f} | {int(r['nb'])} articles | {s}")

### 💬 Commentaire analytique — Q10
> **Complète avec tes observations :**  
> Les médias **[francophones/anglophones/africains]** couvrent le Bénin avec le ton le plus favorable (médiane: **[valeur]**). Les médias **[espace X]** sont plus critiques. Cette différence révèle des biais narratifs liés aux relations historiques et géopolitiques.

---
## 🏆 Synthèse — 5 Insights clés pour la soumission

In [ ]:
print('='*65)
print('  🇧🇯 BÉNIN INSIGHTS 2025 — SYNTHÈSE DATA ANALYSTE')
print('='*65)

sg = df['GoldsteinScale'].mean()
tg = df['AvgTone'].mean()
top_pays_code = df['Actor1CountryCode'].value_counts().index[0]
top_pays_nom = PAYS.get(top_pays_code, top_pays_code)
pct_conflit = (df['EventRootCode']>=13).sum()/len(df)*100
nord = df[df['ActionGeo_FullName'].str.contains('Kandi|Natitingou|Parakou',na=False)]
pct_nord = (nord['EventRootCode']>=13).sum()/len(nord)*100 if len(nord)>0 else 0
pic_mois = df.groupby('mois').size().idxmax()

print(f"""
INSIGHT 1 — STABILITÉ GLOBALE
  Score Goldstein 2025 : {sg:.2f}/10
  → Le Bénin est en zone {'STABLE ✅' if sg>0 else 'INSTABLE ⚠️'} sur 2025.

INSIGHT 2 — COUVERTURE MÉDIATIQUE
  Ton médiatique moyen : {tg:.2f}
  → Image internationale {'POSITIVE 😊' if tg>0 else 'NÉGATIVE 😟'} du Bénin.

INSIGHT 3 — ATTENTION INTERNATIONALE  
  Pays le plus actif : {top_pays_nom}
  → L'attention est concentrée sur quelques acteurs clés.

INSIGHT 4 — FRACTURE NORD/SUD
  Conflits au nord : {pct_nord:.1f}% vs moyenne {pct_conflit:.1f}%
  → Le nord subit une pression sécuritaire documentée.

INSIGHT 5 — PIC MÉDIATIQUE
  Mois le plus couvert : {pic_mois}
  → Coïncide avec un événement politique/sécuritaire majeur.
""")
print('='*65)
print('✅ Notebook complet — Prêt pour soumission GitHub 5 mai 23h59')
print('='*65)

---
## ✅ Checklist soumission

- [ ] Notebook tourne sans erreur (Kernel → Restart & Run All)
- [ ] 5 commentaires analytiques complétés
- [ ] `donnees_benin.csv` dans le même dossier que ce notebook
- [ ] Notebook uploadé sur GitHub
- [ ] Dashboard Streamlit déployé — URL publique obtenue
- [ ] Usage IA mentionné dans le README

---
*Bénin Insights Challenge 2026 — iSHEERO × DataCamp Donates*  
*Data Analyste | Usage IA : Claude (Anthropic)*